# 07 — Sharpened Pipeline: Variance Filter + Random ForestKeep top 20K variance genes, replace LR with RF.

In [ ]:
import pandas as pdimport matplotlib.pyplot as pltfrom gtex_biomarkers.config import Configfrom gtex_biomarkers.data import (    load_raw_data, filter_whole_blood,    build_blood_expression_matrix, build_blood_subjid, variance_filter,)from gtex_biomarkers.labels import discover_tissue_category_pairsfrom gtex_biomarkers.models import make_rf_modelfrom gtex_biomarkers.utils import run_all_tissue_models_parallelfrom gtex_biomarkers.evaluation import (    plot_roc_grid, plot_pr_grid, plot_cm_grid, plot_boxplot_grid,)Config.ensure_dirs()df_expr, df_samples, _, df_meta_url = load_raw_data()blood_meta = filter_whole_blood(df_samples)X_wb, _ = build_blood_expression_matrix(df_expr, blood_meta)blood_subjid = build_blood_subjid(X_wb)

## Variance Filter — Top 20K Genes

In [ ]:
X_wb_var, gene_var = variance_filter(X_wb)print(f"Filtered: {X_wb.shape[1]:,} → {X_wb_var.shape[1]:,} genes")fig, ax = plt.subplots(figsize=(10, 4))ax.hist(gene_var.values, bins=100, color="#2196F3", alpha=0.7, edgecolor="white")ax.axvline(gene_var.iloc[Config.N_TOP_VAR_GENES - 1], color="red", ls="--", lw=1.5,           label=f"Cutoff: var = {gene_var.iloc[Config.N_TOP_VAR_GENES - 1]:.4f}")ax.set_xlabel("Gene variance"); ax.set_ylabel("Count")ax.set_title("Per-gene variance distribution"); ax.legend()fig.savefig(Config.FIGURES_DIR / "gene_variance_distribution.pdf", bbox_inches="tight")plt.show()

## Run RF Models (Parallelized by Tissue)

In [ ]:
pairs_df = discover_tissue_category_pairs(df_meta_url)rf_results, rf_summary = run_all_tissue_models_parallel(    pairs_df, df_meta_url, blood_subjid, X_wb_var, make_rf_model)rf_summary.to_csv(Config.TABLES_DIR / "cv_results_all_tissue_rf.csv", index=False)print(f"Completed {len(rf_results)} RF models")display(rf_summary)

## RF Evaluation Plots

In [ ]:
plot_roc_grid(rf_results,             suptitle="All-Tissue RF — ROC (20K var-filtered)",             save_path=Config.FIGURES_DIR / "roc_all_tissue_rf.pdf")plot_boxplot_grid(rf_results,                  suptitle="All-Tissue RF — Box Plots",                  save_path=Config.FIGURES_DIR / "boxplot_all_tissue_rf.pdf")